In [0]:
#dbutils.fs.rm("/Volumes/data_landing/data_raw/city_time_series/_checkpoints", True)

# this will be uncommented for initial run and 
# for further runs this commented due to following reason
# The Checkpoint "Memory" (Most Likely): Auto Loader uses the checkpointLocation to track which files have been processed. If you ran this code once (even with errors or with a different schema) and it "successfully" acknowledged chunk3.json, it will never process that file again, even if the table is empty or deleted. ONLY UNCOMMENT FOR FIRST RUN OR TESTING and then COMMENT For IDEMPOTENCY.

**1. Architecture Overview**
Auto Loader is designed for incremental and idempotent data ingestion. It uses a mechanism called "Schema Inference" and "Checkpointing" to track which files have been processed.

Why we are using this specific approach:
- Memory Efficiency: By providing an explicit schema (sampling the first row), we prevent the Spark Driver from scanning the entire 511MB file, which would cause an OutOfMemoryError.
- Idempotency: The checkpointLocation ensures that if the job fails and restarts, Spark ignores files that were already successfully written to the Bronze table.
- Uniformity: Every column is cast to String to match existing COPY INTO tables, while load_dt is system-generated as a Timestamp.

**2. Prerequisites & Setup**
Before executing the code, ensure the following configurations are in place:

A. Directory Structure
Ensure your Unity Catalog Volumes are organized as follows:
- Landing (Source): /Volumes/data_landing/data_raw/<dataset_name>/chunks/chunk3.json
- Bronze (Metadata): /Volumes/data_bronze/bronze/checkpoints/<dataset_name>_json/

B. Cluster Settings
- Runtime: Databricks Runtime 13.3 LTS or higher.
- Permissions: Ensure you have READ VOLUME on the landing path and WRITE VOLUME on the bronze path.


In [0]:
import os
import json
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# --- CONFIGURATION ---
VOLUME_PREFIX    = "dbfs:/Volumes/data_landing/data_raw" 
DEST_CATALOG     = "data_bronze"
DEST_SCHEMA      = "bronze"

def process_json_stream(dataset_name, vol_path, target_table):
    """
    Ingests ONLY chunk3.json. 
    Manages checkpoints to ensure fresh processing if data is missing.
    """
    source_folder = f"{vol_path}/chunks"
    # Pointing to the specific dataset checkpoint directory
    checkpoint_root = f"{vol_path}/_checkpoints"
    checkpoint_path = f"{checkpoint_root}/json_ingest_{dataset_name}"
    
    print(f"  - Initializing Auto Loader for {dataset_name}...")

    # --- 1. IDEMPOTENCY & CHECKPOINT CLEANUP ---
    try:
        table_exists = spark.catalog.tableExists(target_table)
        data_already_in_table = False
        
        if table_exists:
            data_already_in_table = spark.table(target_table).filter(F.col("source") == "chunk3.json").limit(1).count() > 0
        
        if data_already_in_table:
            print(f"  [IDEMPOTENT] chunk3.json already exists in {target_table}. Skipping.")
            return
        else:
            # If data is NOT in the table, we must clear the checkpoint memory 
            # to ensure Auto Loader doesn't think it has already "seen" the file.
            print(f"  [CLEANUP] Removing stale checkpoint at: {checkpoint_path}")
            dbutils.fs.rm(checkpoint_path, True)
            
    except Exception as e:
        print(f"  [WARNING] Error during checkpoint/idempotency check: {str(e)}")

    # --- 2. SETUP AUTO LOADER ---
    stream_df = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", checkpoint_path)
        .option("cloudFiles.inferColumnTypes", "false") 
        .option("pathGlobFilter", "chunk3.json") 
        .option("multiline", "true")
        .load(source_folder))

    # --- 3. TRANSFORM & METADATA ---
    data_columns = [c for c in stream_df.columns if c.lower() not in ["load_dt", "source"]]

    final_df = stream_df.select("*", "_metadata.file_path").select(
        *[F.col(c).cast("string") for c in data_columns],
        F.current_timestamp().alias("load_dt"), 
        F.element_at(F.split(F.col("file_path"), "/"), -1).alias("source")
    )

    # --- 4. WRITE STREAM ---
    query = (final_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table))
    
    query.awaitTermination()
    
    # Final Validation
    new_count = spark.table(target_table).filter(F.col("source") == "chunk3.json").count()
    print(f"  - Successfully processed. Current rows from chunk3: {new_count}")

# --- ORCHESTRATION ---
def run_json_ingestion_pipeline():
    datasets_json = dbutils.widgets.get("datasets_json")
    dataset_list = json.loads(datasets_json)
    
    for ds in dataset_list:
        clean_name = ds.lower()
        vol_path_spark = f"{VOLUME_PREFIX}/{clean_name}"
        vol_path_check = f"/Volumes/data_landing/data_raw/{clean_name}"
        full_table_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{clean_name}"

        try:
            # 1. Check if the 'chunks' folder and 'chunk3.json' exist
            files = dbutils.fs.ls(f"{vol_path_check}/chunks")
            if any(f.name == "chunk3.json" for f in files):
                print(f"\n[PROCESSING] {ds}")
                process_json_stream(ds, vol_path_spark, full_table_name)
            else:
                print(f"\n[SKIP] {ds}: chunk3.json not found in chunks folder.")
        except Exception as e:
            if "java.io.FileNotFoundException" in str(e):
                 print(f"\n[SKIP] {ds}: 'chunks' folder hierarchy not found.")
            else:
                 print(f"\n[ERROR] {ds}: {str(e)}")

if __name__ == "__main__":
    run_json_ingestion_pipeline()

**Unit testing:**

The validation script performs a battery of four critical checks for each dataset passed via the datasets_json widget:
- Table Existence: Confirms the Delta table was successfully initialized in the data_bronze.bronze schema.
- Source Specificity: Validates that records originating specifically from chunk3.json are present (proving the pathGlobFilter worked).
- Schema Enforcement: Ensures the "All-String" rule is maintained (all data columns must be StringType).
- Audit Integrity: Confirms load_dt is a valid TimestampType and not a string, allowing for future temporal analysis in the Silver layer.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, TimestampType
import json

def validate_json_ingestion(dataset_name):
    """
    Validates the specific ingestion of chunk3.json for a given dataset.
    Returns a dictionary of results for reporting.
    """
    table_name = f"{DEST_CATALOG}.{DEST_SCHEMA}.{dataset_name.lower()}"
    report = {
        "Dataset": dataset_name,
        "Table": table_name,
        "Status": "SKIPPED",
        "JSON_Rows": 0,
        "Issues": []
    }

    # 1. Check if the table even exists
    if not spark.catalog.tableExists(table_name):
        report["Issues"].append("Table does not exist in Catalog.")
        return report

    try:
        # 2. Filter specifically for the JSON chunk ingested by Auto Loader
        # This prevents confusion if CSV chunks were also ingested earlier
        json_df = spark.table(table_name).filter(F.col("source") == "chunk3.json")
        json_count = json_df.count()
        report["JSON_Rows"] = json_count

        if json_count == 0:
            report["Issues"].append("No records found with source 'chunk3.json'.")
            report["Status"] = "NOT_FOUND"
            return report

        # 3. Schema Validation: All-String (except load_dt)
        # We ensure no numeric types leaked into the Bronze layer
        schema_errors = []
        for field in json_df.schema:
            if field.name == "load_dt":
                if not isinstance(field.dataType, TimestampType):
                    schema_errors.append(f"load_dt is {field.dataType}, expected Timestamp")
            else:
                if not isinstance(field.dataType, StringType):
                    schema_errors.append(f"{field.name} is {field.dataType}, expected String")
        
        if schema_errors:
            report["Issues"].extend(schema_errors)
            report["Status"] = "SCHEMA_FAIL"
        else:
            report["Status"] = "PASSED"

    except Exception as e:
        report["Status"] = "ERROR"
        report["Issues"].append(str(e)[:100])

    return report

# --- MAIN TEST EXECUTION ---

# 1. Fetch the dataset list from the same widget as the ingestion code
datasets_val = dbutils.widgets.get("datasets_json")
target_list = json.loads(datasets_val)

# 2. Collect results
all_reports = []
for ds in target_list:
    all_reports.append(validate_json_ingestion(ds))

# 3. Create a summary DataFrame for visual reporting
report_df = spark.createDataFrame(all_reports)

print("--- BRONZE LAYER AUTO LOADER VALIDATION REPORT ---")
# Order columns for better scannability
display(report_df.select("Status", "Dataset", "JSON_Rows", "Issues", "Table"))

# 4. Optional: Stop the job if critical failures exist
failures = report_df.filter(F.col("Status").isin(["SCHEMA_FAIL", "ERROR"])).count()
if failures > 0:
    print(f"CRITICAL: {failures} datasets failed technical validation.")


# NOTE: # Validation failed for datasets which were not chunked and 2 are chunked so they pass the condition